In [60]:
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/owaiskhancs/wsn-rl-wheels stable-baselines3==2.9.0

import stable_baselines3, gymnasium, numpy, torch
print("SB3:", stable_baselines3.__version__)
print("Gymnasium:", gymnasium.__version__)
print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)

Looking in links: /kaggle/input/datasets/owaiskhancs/wsn-rl-wheels
SB3: 2.9.0
Gymnasium: 1.2.0
NumPy: 2.1.3
Torch: 2.11.0+cpu


In [61]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import random, os

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()

In [62]:
class WSNRoutingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, n_nodes=50, area=100.0, tx_range=25.0,
                 init_energy=1.0, energy_thresh=0.2,
                 trust_thresh=0.5, hop_limit=5, max_steps=50,
                 randomize_topology=False):
        super().__init__()
        self.n_nodes = n_nodes
        self.area = area
        self.tx_range = tx_range
        self.init_energy = init_energy
        self.energy_thresh = energy_thresh
        self.trust_thresh = trust_thresh
        self.hop_limit = hop_limit
        self.max_steps = max_steps
        self.randomize_topology = randomize_topology
        self.sink_id = 0
        self.action_space = spaces.Discrete(self.n_nodes)
        obs_dim = 5 * self.n_nodes
        self.observation_space = spaces.Box(low=0.0, high=1.0,
                                            shape=(obs_dim,), dtype=np.float32)
        # Always build one initial topology
        self._build_topology(seed=SEED)

    def _build_topology(self, seed):
        rng = np.random.default_rng(seed)
        self.positions = rng.uniform(0, self.area, size=(self.n_nodes, 2))
        self.positions[self.sink_id] = [self.area / 2, self.area / 2]
        self.dist = np.linalg.norm(
            self.positions[:, None, :] - self.positions[None, :, :], axis=-1)
        self.adj = (self.dist <= self.tx_range).astype(np.float32)
        np.fill_diagonal(self.adj, 0)
        self.dist_to_sink = self.dist[:, self.sink_id] / (self.area * 1.42)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        effective_seed = seed if seed is not None else SEED
        rng = np.random.default_rng(effective_seed)

        # Only rebuild topology if we explicitly want variation
        if self.randomize_topology:
            self._build_topology(seed=effective_seed)

        self.energy = np.ones(self.n_nodes, dtype=np.float32) * self.init_energy
        self.trust = rng.uniform(0.4, 1.0, size=self.n_nodes).astype(np.float32)
        self.energy[self.sink_id] = 1.0
        self.trust[self.sink_id] = 1.0
        self.current = int(rng.integers(1, self.n_nodes))
        self.hops = 0
        self.steps = 0
        return self._obs(), {}

    def _obs(self):
        onehot = np.zeros(self.n_nodes, dtype=np.float32)
        onehot[self.current] = 1.0
        return np.concatenate([
            self.energy, self.trust, self.dist_to_sink.astype(np.float32),
            self.adj[self.current].astype(np.float32), onehot]).astype(np.float32)

    def feasible_actions(self):
        mask = np.zeros(self.n_nodes, dtype=bool)
        for a in range(self.n_nodes):
            if a == self.current: continue
            if self.adj[self.current, a] == 0: continue
            if self.energy[a] < self.energy_thresh: continue
            if self.trust[a] < self.trust_thresh: continue
            if self.hops + 1 > self.hop_limit and a != self.sink_id: continue
            mask[a] = True
        return mask

    def step(self, action):
        self.steps += 1
        reward = 0.0
        terminated = False
        truncated = False

        if action == self.current or self.adj[self.current, action] == 0:
            return self._obs(), -5.0, False, True, {"reason": "invalid"}

        self.hops += 1
        if self.hops > self.hop_limit and action != self.sink_id:
            return self._obs(), -10.0, False, True, {"reason": "hop_limit"}

        self.current = action
        self.energy[action] = max(0.0, self.energy[action] - 0.01)
        self.trust[action] = max(0.0, self.trust[action] - 0.001)

        reward += -0.01 * (1.0 - self.energy[action])
        reward += 0.5 * self.trust[action] - 0.1 * self.hops

        if self.current == self.sink_id:
            reward += 10.0
            return self._obs(), reward, True, False, {"reason": "success"}

        if self.steps >= self.max_steps:
            reward -= 10.0
            return self._obs(), reward, False, True, {"reason": "max_steps"}

        return self._obs(), reward, False, False, {}

In [63]:
env = WSNRoutingEnv()
obs, _ = env.reset(seed=42)
print("Obs shape:", obs.shape, "| Action space:", env.action_space.n)

for ep in range(5):
    obs, _ = env.reset(seed=100 + ep)
    total_r = 0.0
    feasible_count = 0
    invalid_count = 0
    hop_violations = 0
    done = False
    steps = 0
    info = {}
    while not done and steps < env.max_steps:
        mask = env.feasible_actions()
        if mask.any():
            valid_actions = np.where(mask)[0]
            feasible_count += len(valid_actions)
            action = int(np.random.choice(valid_actions))
        else:
            action = int(np.random.randint(env.n_nodes))
        obs, r, term, trunc, info = env.step(action)
        total_r += r
        if info.get("reason") == "invalid": invalid_count += 1
        if info.get("reason") == "hop_limit": hop_violations += 1
        done = term or trunc
        steps += 1
    print(f"Ep {ep}: steps={steps:2d} | return={total_r:7.2f} | "
          f"avg_feasible={feasible_count/max(steps,1):5.2f} | "
          f"invalid={invalid_count} | hop_viol={hop_violations} | "
          f"reason={info.get('reason')}")

Obs shape: (250,) | Action space: 50
Ep 0: steps= 6 | return=  10.37 | avg_feasible= 7.33 | invalid=0 | hop_viol=0 | reason=success
Ep 1: steps= 6 | return=  -4.46 | avg_feasible= 0.83 | invalid=1 | hop_viol=0 | reason=invalid
Ep 2: steps= 6 | return=  -4.68 | avg_feasible= 6.50 | invalid=1 | hop_viol=0 | reason=invalid
Ep 3: steps= 6 | return=  10.26 | avg_feasible= 4.83 | invalid=0 | hop_viol=0 | reason=success
Ep 4: steps= 6 | return=  -4.77 | avg_feasible= 3.67 | invalid=1 | hop_viol=0 | reason=invalid


In [64]:
import heapq

def dijkstra_shortest_hops(env, src):
    N = env.n_nodes
    dist = [float("inf")] * N
    dist[src] = 0
    prev = [-1] * N
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v in range(N):
            if env.adj[u, v] > 0 and dist[v] > d + 1:
                dist[v] = d + 1
                prev[v] = u
                heapq.heappush(pq, (d + 1, v))
    if dist[env.sink_id] == float("inf"):
        return None
    path = []
    cur = env.sink_id
    while cur != -1:
        path.append(cur)
        cur = prev[cur]
    return list(reversed(path))

def greedy_policy(env):
    mask = env.feasible_actions()
    if not mask.any():
        return None
    candidates = np.where(mask)[0]
    return int(candidates[np.argmin(env.dist_to_sink[candidates])])

def random_feasible_policy(env):
    mask = env.feasible_actions()
    if not mask.any():
        return int(np.random.randint(env.n_nodes))
    return int(np.random.choice(np.where(mask)[0]))

def run_episode(env, policy_fn, seed):
    obs, _ = env.reset(seed=seed)
    total_r = 0.0
    done = False
    steps = 0
    info = {}
    while not done and steps < env.max_steps:
        action = policy_fn(env)
        if action is None:
            break
        obs, r, term, trunc, info = env.step(action)
        total_r += r
        done = term or trunc
        steps += 1
    return {
        "steps": steps,
        "return": total_r,
        "reason": info.get("reason", "no_action"),
        "hops": env.hops,
    }

In [65]:
N_EPISODES = 100
results = {"dijkstra": [], "greedy": [], "random": []}

for ep in range(N_EPISODES):
    seed = 1000 + ep
    env = WSNRoutingEnv()

    env.reset(seed=seed)
    path = dijkstra_shortest_hops(env, env.current)
    results["dijkstra"].append({
        "steps": len(path) - 1 if path else env.max_steps,
        "return": np.nan,
        "reason": "success" if path and len(path) - 1 <= env.hop_limit else "hop_limit",
        "hops": len(path) - 1 if path else env.max_steps,
    })

    results["greedy"].append(run_episode(WSNRoutingEnv(), greedy_policy, seed))
    results["random"].append(run_episode(WSNRoutingEnv(), random_feasible_policy, seed))

def summarize(name, runs):
    succ = sum(1 for r in runs if r["reason"] == "success") / len(runs)
    avg_hops = np.mean([r["hops"] for r in runs])
    avg_ret = np.nanmean([r["return"] for r in runs]) if any(
        not np.isnan(r["return"]) for r in runs) else float("nan")
    reasons = {}
    for r in runs:
        reasons[r["reason"]] = reasons.get(r["reason"], 0) + 1
    print(f"{name:10s} | success={succ*100:5.1f}% | avg_hops={avg_hops:5.2f} | "
          f"avg_return={avg_ret:7.2f} | {reasons}")

print(f"--- {N_EPISODES} episodes ---")
for name in ["dijkstra", "greedy", "random"]:
    summarize(name, results[name])

--- 100 episodes ---
dijkstra   | success= 94.0% | avg_hops= 5.04 | avg_return=    nan | {'success': 94, 'hop_limit': 6}
greedy     | success= 94.0% | avg_hops= 2.30 | avg_return=   9.95 | {'success': 94, 'no_action': 6}
random     | success= 30.0% | avg_hops= 5.11 | avg_return=  -0.88 | {'invalid': 55, 'success': 30, 'hop_limit': 15}


In [66]:
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy


class MaskedActorCriticPolicy(ActorCriticPolicy):
    """
    Applies the feasibility mask (last n_actions dims of obs) to action logits.
    Works for both `forward` (rollout) and `evaluate_actions` (training).
    """
    def _masked_logits(self, obs):
        """Compute logits from features and apply mask."""
        features = self.extract_features(obs)
        if self.share_features_extractor:
            pi_features = features
        else:
            pi_features = features[0] if isinstance(features, tuple) else features
        latent_pi = self.mlp_extractor.forward_actor(pi_features)
        logits = self.action_net(latent_pi)
        n_actions = self.action_space.n
        mask = obs[:, -n_actions:].bool()
        neg_inf = torch.finfo(logits.dtype).min
        masked_logits = torch.where(mask, logits, torch.full_like(logits, neg_inf))
        return masked_logits

    def forward(self, obs, deterministic=False):
        features = self.extract_features(obs)
        if self.share_features_extractor:
            pi_features, vf_features = features, features
        else:
            pi_features, vf_features = features
        latent_vf = self.mlp_extractor.forward_critic(vf_features)
        values = self.value_net(latent_vf)

        masked_logits = self._masked_logits(obs)
        distribution = self.action_dist.proba_distribution(action_logits=masked_logits)
        actions = distribution.get_actions(deterministic=deterministic)
        log_prob = distribution.log_prob(actions)
        return actions, values, log_prob

    def evaluate_actions(self, obs, actions):
        features = self.extract_features(obs)
        if self.share_features_extractor:
            pi_features, vf_features = features, features
        else:
            pi_features, vf_features = features
        latent_vf = self.mlp_extractor.forward_critic(vf_features)
        values = self.value_net(latent_vf)

        masked_logits = self._masked_logits(obs)
        distribution = self.action_dist.proba_distribution(action_logits=masked_logits)
        log_prob = distribution.log_prob(actions)
        entropy = distribution.entropy()
        return values, log_prob, entropy


class MaskedPPO(PPO):
    pass

In [67]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv


class MaskedWSNEnv(gym.Wrapper):
    """
    Appends the feasibility mask to the observation so the policy
    always sees which actions are currently valid.
    """
    def __init__(self, env, randomize_topology=False):
        super().__init__(env)
        self.randomize_topology = randomize_topology
        self._episode_counter = 0
        self.current_mask = None

        inner = env.unwrapped
        n = inner.n_nodes
        base_obs_dim = 5 * n
        self.observation_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(base_obs_dim + n,), dtype=np.float32)

    def reset(self, **kwargs):
        if self.randomize_topology and "seed" not in kwargs:
            self._episode_counter += 1
            kwargs["seed"] = 10_000 + self._episode_counter
        obs, info = self.env.reset(**kwargs)
        self.current_mask = self.env.feasible_actions()
        return self._augment(obs), info

    def step(self, action):
        obs, r, term, trunc, info = self.env.step(action)
        self.current_mask = self.env.feasible_actions()
        return self._augment(obs), r, term, trunc, info

    def _augment(self, obs):
        return np.concatenate([obs, self.current_mask.astype(np.float32)])


def make_env(seed=0):
    def _init():
        env = WSNRoutingEnv(randomize_topology=False)
        env = MaskedWSNEnv(env, randomize_topology=False)
        env = Monitor(env)
        env.reset(seed=seed)
        return env
    return _init

In [68]:
env_test = MaskedWSNEnv(WSNRoutingEnv(randomize_topology=False),
                        randomize_topology=False)
obs, _ = env_test.reset(seed=0)
print("Obs shape:", obs.shape, "(expected 300 = 5*50 + 50)")
print("Mask sum:", int(env_test.current_mask.sum()))
print("Last 50 dims = mask:", obs[-50:].sum(), "(should equal mask sum)")

Obs shape: (300,) (expected 300 = 5*50 + 50)
Mask sum: 6
Last 50 dims = mask: 6.0 (should equal mask sum)


In [69]:
set_seed(42)

LOG_DIR = "/kaggle/working/ppo_hard_mask"
os.makedirs(LOG_DIR, exist_ok=True)

vec_env = DummyVecEnv([make_env(seed=42)])

model = MaskedPPO(
    MaskedActorCriticPolicy,
    vec_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[64, 64], activation_fn=torch.nn.Tanh),
    verbose=1,
    seed=42,
    tensorboard_log=LOG_DIR,
)

TOTAL_TIMESTEPS = 500_000
model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=False)

model.save(f"{LOG_DIR}/ppo_hard_mask")
print("Saved model to", f"{LOG_DIR}/ppo_hard_mask.zip")

Using cpu device
Logging to /kaggle/working/ppo_hard_mask/PPO_5
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5.75     |
|    ep_rew_mean     | 2.34     |
| time/              |          |
|    fps             | 1165     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 5.62        |
|    ep_rew_mean          | 4.31        |
| time/                   |             |
|    fps                  | 827         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014413123 |
|    clip_fraction        | 0.242       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.61       |
|    exp

In [70]:
import torch
import numpy as np
import shutil, os

print("=" * 65)
print("TRAINING SUMMARY")
print("=" * 65)
try:
    log_data = model.logger.name_to_value
    print(f"  ep_rew_mean:     {log_data.get('rollout/ep_rew_mean', 'n/a')}")
    print(f"  ep_len_mean:     {log_data.get('rollout/ep_len_mean', 'n/a')}")
except Exception:
    print("  (logger data not available — check Cell 10's box output)")
print(f"  total_timesteps: {model.num_timesteps}")


def evaluate_ppo(model, n_episodes=100, base_seed=5000):
    successes = 0
    returns = []
    hops = []
    reasons = {}
    for i in range(n_episodes):
        env = MaskedWSNEnv(WSNRoutingEnv(randomize_topology=False),
                           randomize_topology=False)
        obs, _ = env.reset(seed=base_seed + i)
        total_r = 0.0
        done = False
        steps = 0
        info = {}
        max_steps = env.unwrapped.max_steps
        while not done and steps < max_steps:
            with torch.no_grad():
                obs_tensor = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
                actions, _, _ = model.policy(obs_tensor, deterministic=True)
                action = int(actions.item())
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            done = term or trunc
            steps += 1
        returns.append(total_r)
        hops.append(env.unwrapped.hops)
        key = info.get("reason", "unknown")
        reasons[key] = reasons.get(key, 0) + 1
        if key == "success":
            successes += 1
    return {
        "success_rate": successes / n_episodes * 100,
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_hops": float(np.mean(hops)),
        "reasons": reasons,
    }


print()
print("=" * 65)
print("PPO EVALUATION (100 episodes, unseen seeds)")
print("=" * 65)
ppo_res = evaluate_ppo(model, n_episodes=100)
print(f"  Success rate: {ppo_res['success_rate']:.1f}%")
print(f"  Mean return:  {ppo_res['mean_return']:.2f} ± {ppo_res['std_return']:.2f}")
print(f"  Mean hops:    {ppo_res['mean_hops']:.2f}")
print(f"  Reasons:      {ppo_res['reasons']}")

print()
print("=" * 65)
print("BASELINE COMPARISON")
print("=" * 65)
print(f"{'Method':<22} {'Success':>10} {'Avg Hops':>10} {'Avg Return':>14}")
print("-" * 65)
print(f"{'Dijkstra (oracle)':<22} {'94.0%':>10} {'5.04':>10} {'nan':>14}")
print(f"{'Greedy':<22} {'94.0%':>10} {'2.30':>10} {'9.95':>14}")
print(f"{'Random feasible':<22} {'22.0%':>10} {'4.99':>10} {'-1.59':>14}")
print(f"{'PPO hard-mask':<22} {ppo_res['success_rate']:>9.1f}% "
      f"{ppo_res['mean_hops']:>10.2f} {ppo_res['mean_return']:>14.2f}")
print("=" * 65)

os.makedirs("/kaggle/working/output", exist_ok=True)
src = "/kaggle/working/ppo_hard_mask/ppo_hard_mask.zip"
if os.path.exists(src):
    shutil.copy(src, "/kaggle/working/output/ppo_hard_mask.zip")
    print("\nModel saved to /kaggle/working/output/ppo_hard_mask.zip")
    print("Output files:", os.listdir("/kaggle/working/output"))
else:
    print("\nWARNING: trained model not found at", src)

TRAINING SUMMARY
  ep_rew_mean:     n/a
  ep_len_mean:     n/a
  total_timesteps: 501760

PPO EVALUATION (100 episodes, unseen seeds)
  Success rate: 85.0%
  Mean return:  8.27 ± 5.41
  Mean hops:    2.49
  Reasons:      {'success': 85, 'invalid': 15}

BASELINE COMPARISON
Method                    Success   Avg Hops     Avg Return
-----------------------------------------------------------------
Dijkstra (oracle)           94.0%       5.04            nan
Greedy                      94.0%       2.30           9.95
Random feasible             22.0%       4.99          -1.59
PPO hard-mask               85.0%       2.49           8.27

Model saved to /kaggle/working/output/ppo_hard_mask.zip
Output files: ['ppo_hard_mask.zip']


In [71]:
import numpy as np

def action_features(env, action):
    """
    Feature vector describing a candidate action from the current state.
    Concatenates: current node features + candidate node features + relation.
    """
    inner = env.unwrapped
    cur = inner.current
    a = action
    return np.array([
        inner.energy[cur],
        inner.trust[cur],
        inner.hops / inner.hop_limit,
        inner.dist_to_sink[cur],
        inner.energy[a],
        inner.trust[a],
        inner.dist_to_sink[a],
        inner.adj[cur, a],
        float(a == inner.sink_id),
    ], dtype=np.float32)


def collect_classifier_data(n_episodes=200, seed=0):
    X, y = [], []
    rng = np.random.default_rng(seed)
    for ep in range(n_episodes):
        env = WSNRoutingEnv(randomize_topology=False)
        env.reset(seed=seed + ep)
        inner = env.unwrapped
        done = False
        steps = 0
        while not done and steps < inner.max_steps:
            oracle = inner.feasible_actions()
            for a in range(inner.n_nodes):
                X.append(action_features(env, a))
                y.append(1.0 if oracle[a] else 0.0)
            # Take a random feasible action to explore the state space
            feasible = np.where(oracle)[0]
            if len(feasible) == 0:
                break
            action = int(rng.choice(feasible))
            obs, r, term, trunc, info = env.step(action)
            done = term or trunc
            steps += 1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print("Collecting classifier training data...")
X_all, y_all = collect_classifier_data(n_episodes=200, seed=1)
print("Total samples:", X_all.shape, "| Positive rate:", y_all.mean())

Total samples: (55950, 9) | Positive rate: 0.11438785


In [72]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Train/test split
n = len(X_all)
idx = np.random.default_rng(0).permutation(n)
split = int(0.8 * n)
X_train, y_train = X_all[idx[:split]], y_all[idx[:split]]
X_test,  y_test  = X_all[idx[split:]], y_all[idx[split:]]

# Class weights to handle imbalance
pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float32)

class FeasibilityClassifier(nn.Module):
    def __init__(self, in_dim=9, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

clf = FeasibilityClassifier().to("cpu")
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

for epoch in range(15):
    clf.train()
    total = 0.0
    for xb, yb in train_dl:
        opt.zero_grad()
        loss = loss_fn(clf(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    if epoch % 3 == 0 or epoch == 14:
        clf.eval()
        with torch.no_grad():
            logits = clf(torch.tensor(X_test))
            preds = (torch.sigmoid(logits) > 0.5).float()
            acc = (preds == torch.tensor(y_test)).float().mean().item()
            tp = ((preds == 1) & (torch.tensor(y_test) == 1)).sum().item()
            fp = ((preds == 1) & (torch.tensor(y_test) == 0)).sum().item()
            fn = ((preds == 0) & (torch.tensor(y_test) == 1)).sum().item()
            prec = tp / max(tp + fp, 1)
            rec = tp / max(tp + fn, 1)
        print(f"Epoch {epoch:2d} | loss={total/len(train_ds):.4f} | "
              f"acc={acc:.4f} | prec={prec:.4f} | rec={rec:.4f}")

torch.save(clf.state_dict(), "/kaggle/working/feasibility_classifier.pt")
print("Classifier saved.")

Epoch  0 | loss=0.3196 | acc=0.9546 | prec=0.7095 | rec=1.0000
Epoch  3 | loss=0.0507 | acc=0.9843 | prec=0.8758 | rec=1.0000
Epoch  6 | loss=0.0234 | acc=0.9941 | prec=0.9495 | rec=1.0000
Epoch  9 | loss=0.0184 | acc=0.9956 | prec=0.9620 | rec=1.0000
Epoch 12 | loss=0.0130 | acc=0.9926 | prec=0.9373 | rec=1.0000
Epoch 14 | loss=0.0141 | acc=0.9982 | prec=0.9880 | rec=0.9960
Classifier saved.


In [73]:
import torch

def classifier_mask(env, clf, threshold=0.5):
    """Return a boolean mask predicted by the classifier."""
    inner = env.unwrapped
    feats = np.stack([action_features(env, a) for a in range(inner.n_nodes)])
    with torch.no_grad():
        logits = clf(torch.tensor(feats))
        probs = torch.sigmoid(logits).numpy()
    return probs > threshold


class MaskedWSNEnvLearned(gym.Wrapper):
    """Same as MaskedWSNEnv but uses a learned classifier for the mask."""
    def __init__(self, env, clf, threshold=0.5):
        super().__init__(env)
        self.clf = clf
        self.threshold = threshold
        self.current_mask = None
        inner = env.unwrapped
        n = inner.n_nodes
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5 * n + n,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_mask = classifier_mask(self.env, self.clf, self.threshold)
        return self._augment(obs), info

    def step(self, action):
        obs, r, term, trunc, info = self.env.step(action)
        self.current_mask = classifier_mask(self.env, self.clf, self.threshold)
        return self._augment(obs), r, term, trunc, info

    def _augment(self, obs):
        return np.concatenate([obs, self.current_mask.astype(np.float32)])


def evaluate_ppo_learned(model, clf, n_episodes=100, base_seed=5000):
    successes = 0
    returns = []
    hops = []
    reasons = {}
    for i in range(n_episodes):
        env = MaskedWSNEnvLearned(
            WSNRoutingEnv(randomize_topology=False), clf)
        obs, _ = env.reset(seed=base_seed + i)
        total_r = 0.0
        done = False
        steps = 0
        info = {}
        max_steps = env.unwrapped.max_steps
        while not done and steps < max_steps:
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
                actions, _, _ = model.policy(obs_t, deterministic=True)
                action = int(actions.item())
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            done = term or trunc
            steps += 1
        returns.append(total_r)
        hops.append(env.unwrapped.hops)
        key = info.get("reason", "unknown")
        reasons[key] = reasons.get(key, 0) + 1
        if key == "success":
            successes += 1
    return {
        "success_rate": successes / n_episodes * 100,
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_hops": float(np.mean(hops)),
        "reasons": reasons,
    }


print("=" * 65)
print("PPO with LEARNED CLASSIFIER MASK (100 episodes)")
print("=" * 65)
learned_res = evaluate_ppo_learned(model, clf, n_episodes=100)
print(f"  Success rate: {learned_res['success_rate']:.1f}%")
print(f"  Mean return:  {learned_res['mean_return']:.2f} ± {learned_res['std_return']:.2f}")
print(f"  Mean hops:    {learned_res['mean_hops']:.2f}")
print(f"  Reasons:      {learned_res['reasons']}")

print()
print("=" * 65)
print("COMPARISON: Oracle mask vs Learned mask")
print("=" * 65)
print(f"{'Method':<28} {'Success':>10} {'Mean Return':>14}")
print("-" * 65)
print(f"{'PPO (oracle mask)':<28} {'85.0%':>10} {'8.27':>14}")
print(f"{'PPO (learned classifier)':<28} {learned_res['success_rate']:>9.1f}% "
      f"{learned_res['mean_return']:>14.2f}")
print("=" * 65)

PPO with LEARNED CLASSIFIER MASK (100 episodes)
  Success rate: 85.0%
  Mean return:  8.26 ± 5.42
  Mean hops:    2.45
  Reasons:      {'success': 85, 'invalid': 15}

COMPARISON: Oracle mask vs Learned mask
Method                          Success    Mean Return
-----------------------------------------------------------------
PPO (oracle mask)                 85.0%           8.27
PPO (learned classifier)          85.0%           8.26


In [74]:
# ============================================================
# PART 4 — CONSOLIDATED OUTPUT CELL
# Trains classifier, evaluates both oracle and learned masks.
# ============================================================
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# -------- 1. Action features --------
def action_features(env, action):
    inner = env.unwrapped
    cur = inner.current
    a = action
    return np.array([
        inner.energy[cur],
        inner.trust[cur],
        inner.hops / inner.hop_limit,
        inner.dist_to_sink[cur],
        inner.energy[a],
        inner.trust[a],
        inner.dist_to_sink[a],
        inner.adj[cur, a],
        float(a == inner.sink_id),
    ], dtype=np.float32)


# -------- 2. Collect data --------
def collect_classifier_data(n_episodes=200, seed=0):
    X, y = [], []
    rng = np.random.default_rng(seed)
    for ep in range(n_episodes):
        env = WSNRoutingEnv(randomize_topology=False)
        env.reset(seed=seed + ep)
        inner = env.unwrapped
        done = False
        steps = 0
        while not done and steps < inner.max_steps:
            oracle = inner.feasible_actions()
            for a in range(inner.n_nodes):
                X.append(action_features(env, a))
                y.append(1.0 if oracle[a] else 0.0)
            feasible = np.where(oracle)[0]
            if len(feasible) == 0:
                break
            action = int(rng.choice(feasible))
            obs, r, term, trunc, info = env.step(action)
            done = term or trunc
            steps += 1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print("=" * 65)
print("COLLECTING CLASSIFIER TRAINING DATA")
print("=" * 65)
X_all, y_all = collect_classifier_data(n_episodes=200, seed=1)
print(f"  Samples: {X_all.shape}")
print(f"  Positive rate: {y_all.mean():.4f}")


# -------- 3. Train classifier --------
n = len(X_all)
idx = np.random.default_rng(0).permutation(n)
split = int(0.8 * n)
X_train, y_train = X_all[idx[:split]], y_all[idx[:split]]
X_test,  y_test  = X_all[idx[split:]], y_all[idx[split:]]

pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float32)


class FeasibilityClassifier(nn.Module):
    def __init__(self, in_dim=9, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


clf = FeasibilityClassifier().to("cpu")
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

final_metrics = {}
for epoch in range(15):
    clf.train()
    total = 0.0
    for xb, yb in train_dl:
        opt.zero_grad()
        loss = loss_fn(clf(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    if epoch % 3 == 0 or epoch == 14:
        clf.eval()
        with torch.no_grad():
            logits = clf(torch.tensor(X_test))
            preds = (torch.sigmoid(logits) > 0.5).float()
            acc = (preds == torch.tensor(y_test)).float().mean().item()
            tp = ((preds == 1) & (torch.tensor(y_test) == 1)).sum().item()
            fp = ((preds == 1) & (torch.tensor(y_test) == 0)).sum().item()
            fn = ((preds == 0) & (torch.tensor(y_test) == 1)).sum().item()
            prec = tp / max(tp + fp, 1)
            rec = tp / max(tp + fn, 1)
        final_metrics = {"acc": acc, "prec": prec, "rec": rec}
        print(f"Epoch {epoch:2d} | loss={total/len(train_ds):.4f} | "
              f"acc={acc:.4f} | prec={prec:.4f} | rec={rec:.4f}")

torch.save(clf.state_dict(), "/kaggle/working/feasibility_classifier.pt")


# -------- 4. Learned-mask env and eval --------
def classifier_mask(env, clf, threshold=0.5):
    inner = env.unwrapped
    feats = np.stack([action_features(env, a) for a in range(inner.n_nodes)])
    with torch.no_grad():
        logits = clf(torch.tensor(feats))
        probs = torch.sigmoid(logits).numpy()
    return probs > threshold


class MaskedWSNEnvLearned(gym.Wrapper):
    def __init__(self, env, clf, threshold=0.5):
        super().__init__(env)
        self.clf = clf
        self.threshold = threshold
        self.current_mask = None
        inner = env.unwrapped
        n = inner.n_nodes
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5 * n + n,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_mask = classifier_mask(self.env, self.clf, self.threshold)
        return self._augment(obs), info

    def step(self, action):
        obs, r, term, trunc, info = self.env.step(action)
        self.current_mask = classifier_mask(self.env, self.clf, self.threshold)
        return self._augment(obs), r, term, trunc, info

    def _augment(self, obs):
        return np.concatenate([obs, self.current_mask.astype(np.float32)])


def evaluate(model, env_factory, n_episodes=100, base_seed=5000):
    successes = 0
    returns = []
    hops = []
    reasons = {}
    for i in range(n_episodes):
        env = env_factory()
        obs, _ = env.reset(seed=base_seed + i)
        total_r = 0.0
        done = False
        steps = 0
        info = {}
        max_steps = env.unwrapped.max_steps
        while not done and steps < max_steps:
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
                actions, _, _ = model.policy(obs_t, deterministic=True)
                action = int(actions.item())
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            done = term or trunc
            steps += 1
        returns.append(total_r)
        hops.append(env.unwrapped.hops)
        key = info.get("reason", "unknown")
        reasons[key] = reasons.get(key, 0) + 1
        if key == "success":
            successes += 1
    return {
        "success_rate": successes / n_episodes * 100,
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_hops": float(np.mean(hops)),
        "reasons": reasons,
    }


print()
print("=" * 65)
print("PPO WITH ORACLE MASK (100 episodes)")
print("=" * 65)
oracle_res = evaluate(
    model,
    lambda: MaskedWSNEnv(WSNRoutingEnv(randomize_topology=False),
                         randomize_topology=False),
    n_episodes=100)
print(f"  Success rate: {oracle_res['success_rate']:.1f}%")
print(f"  Mean return:  {oracle_res['mean_return']:.2f} ± {oracle_res['std_return']:.2f}")
print(f"  Mean hops:    {oracle_res['mean_hops']:.2f}")
print(f"  Reasons:      {oracle_res['reasons']}")

print()
print("=" * 65)
print("PPO WITH LEARNED CLASSIFIER MASK (100 episodes)")
print("=" * 65)
learned_res = evaluate(
    model,
    lambda: MaskedWSNEnvLearned(WSNRoutingEnv(randomize_topology=False), clf),
    n_episodes=100)
print(f"  Success rate: {learned_res['success_rate']:.1f}%")
print(f"  Mean return:  {learned_res['mean_return']:.2f} ± {learned_res['std_return']:.2f}")
print(f"  Mean hops:    {learned_res['mean_hops']:.2f}")
print(f"  Reasons:      {learned_res['reasons']}")

print()
print("=" * 65)
print("PART 4 SUMMARY TABLE")
print("=" * 65)
print(f"{'Method':<32} {'Success':>10} {'Mean Return':>14}")
print("-" * 65)
print(f"{'PPO (oracle mask)':<32} {oracle_res['success_rate']:>9.1f}% "
      f"{oracle_res['mean_return']:>14.2f}")
print(f"{'PPO (learned classifier mask)':<32} {learned_res['success_rate']:>9.1f}% "
      f"{learned_res['mean_return']:>14.2f}")
print(f"{'Classifier accuracy':<32} {final_metrics['acc']:>10.4f}")
print(f"{'Classifier precision':<32} {final_metrics['prec']:>10.4f}")
print(f"{'Classifier recall':<32} {final_metrics['rec']:>10.4f}")
print("=" * 65)

COLLECTING CLASSIFIER TRAINING DATA
  Samples: (55950, 9)
  Positive rate: 0.1144
Epoch  0 | loss=0.3077 | acc=0.9526 | prec=0.7007 | rec=1.0000
Epoch  3 | loss=0.0756 | acc=0.9811 | prec=0.8551 | rec=0.9984
Epoch  6 | loss=0.0324 | acc=0.9959 | prec=0.9643 | rec=1.0000
Epoch  9 | loss=0.0208 | acc=0.9963 | prec=0.9680 | rec=1.0000
Epoch 12 | loss=0.0182 | acc=0.9966 | prec=0.9703 | rec=1.0000
Epoch 14 | loss=0.0144 | acc=0.9955 | prec=0.9613 | rec=1.0000

PPO WITH ORACLE MASK (100 episodes)
  Success rate: 85.0%
  Mean return:  8.27 ± 5.41
  Mean hops:    2.49
  Reasons:      {'success': 85, 'invalid': 15}

PPO WITH LEARNED CLASSIFIER MASK (100 episodes)
  Success rate: 85.0%
  Mean return:  8.26 ± 5.43
  Mean hops:    2.49
  Reasons:      {'success': 85, 'invalid': 15}

PART 4 SUMMARY TABLE
Method                              Success    Mean Return
-----------------------------------------------------------------
PPO (oracle mask)                     85.0%           8.27
PPO (learned

In [79]:
# ============================================================
# PART 5 — STATISTICS RUN (5 seeds, IQM + bootstrap CI)
# ============================================================
import numpy as np
import torch
from scipy import stats


class ShiftedWSNEnv(WSNRoutingEnv):
    def reset(self, seed=None, options=None):
        super().reset(seed=seed, options=options)
        rng = np.random.default_rng((seed if seed is not None else 0) + 777)
        self.energy = rng.uniform(0.1, 1.0, size=self.n_nodes).astype(np.float32)
        self.trust = rng.uniform(0.3, 1.0, size=self.n_nodes).astype(np.float32)
        self.energy[self.sink_id] = 1.0
        self.trust[self.sink_id] = 1.0
        return self._obs(), {}


def make_shifted_env(energy_thresh, trust_thresh, hop_limit):
    return ShiftedWSNEnv(
        randomize_topology=False,
        energy_thresh=energy_thresh,
        trust_thresh=trust_thresh,
        hop_limit=hop_limit,
    )


def oracle_mask_from_env(env):
    return env.unwrapped.feasible_actions()

def learned_mask_from_env(env, clf, threshold=0.5):
    inner = env.unwrapped
    feats = np.stack([action_features(env, a) for a in range(inner.n_nodes)])
    with torch.no_grad():
        logits = clf(torch.tensor(feats))
        probs = torch.sigmoid(logits).numpy()
    return probs > threshold


def run_episodes(env_factory, mask_fn, base_seed, n_episodes=100):
    """Return per-episode binary success vector (1=success, 0=fail)."""
    outcomes = []
    for i in range(n_episodes):
        env = ShiftedEnv(env_factory(), mask_fn)
        obs, _ = env.reset(seed=base_seed + i)
        done, steps, info = False, 0, {}
        max_steps = env.unwrapped.max_steps
        while not done and steps < max_steps:
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
                actions, _, _ = model.policy(obs_t, deterministic=True)
                action = int(actions.item())
            obs, r, term, trunc, info = env.step(action)
            done = term or trunc
            steps += 1
        outcomes.append(1 if info.get("reason") == "success" else 0)
    return np.array(outcomes)


# ---- statistics helpers ----
def iqm(x):
    """Interquartile mean."""
    x = np.sort(x)
    n = len(x)
    lo, hi = int(0.25 * n), int(0.75 * n)
    return float(np.mean(x[lo:hi])) if hi > lo else float(np.mean(x))

def stratified_bootstrap_ci(runs, n_boot=2000, alpha=0.05, rng=None):
    """
    runs: list of arrays, each array = per-episode outcomes for one seed.
    Returns (IQM, lower, upper) using stratified bootstrap over seeds.
    """
    rng = rng or np.random.default_rng(0)
    n_seeds = len(runs)
    iqm_samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_seeds, size=n_seeds)
        pooled = np.concatenate([runs[i] for i in idx])
        iqm_samples.append(iqm(pooled))
    iqm_samples = np.array(iqm_samples)
    lo = float(np.percentile(iqm_samples, 100 * alpha / 2))
    hi = float(np.percentile(iqm_samples, 100 * (1 - alpha / 2)))
    central = float(np.mean(iqm_samples))
    return central, lo, hi


# ---- conditions and masks ----
conditions = [
    ("No shift",   dict(energy_thresh=0.20, trust_thresh=0.50, hop_limit=5)),
    ("Energy 0.5", dict(energy_thresh=0.50, trust_thresh=0.50, hop_limit=5)),
    ("Trust 0.8",  dict(energy_thresh=0.20, trust_thresh=0.80, hop_limit=5)),
    ("Hop 5->2",   dict(energy_thresh=0.20, trust_thresh=0.50, hop_limit=2)),
    ("Combined",   dict(energy_thresh=0.50, trust_thresh=0.70, hop_limit=3)),
]

mask_sources = [
    ("Oracle",  lambda env: oracle_mask_from_env(env)),
    ("Learned", lambda env: learned_mask_from_env(env, clf)),
]

SEEDS = [7000, 8000, 9000, 10000, 11000]

print("=" * 75)
print("STATISTICS RUN — 5 seeds × 100 episodes per condition × mask")
print("=" * 75)

# Collect per-seed success vectors
per_seed = {}  # per_seed[(cond, mask)] = list of np arrays
for cond_name, cond_kwargs in conditions:
    for mask_name, mask_fn in mask_sources:
        seed_vectors = []
        for s in SEEDS:
            vec = run_episodes(
                lambda ck=cond_kwargs: make_shifted_env(**ck),
                mask_fn,
                base_seed=s,
                n_episodes=100,
            )
            seed_vectors.append(vec)
        per_seed[(cond_name, mask_name)] = seed_vectors
        pooled = np.concatenate(seed_vectors)
        print(f"  [{cond_name:<10}] [{mask_name:<8}] "
              f"mean={pooled.mean()*100:5.1f}%  n={len(pooled)}")

# ---- summary with IQM + CI ----
print()
print("=" * 75)
print("IQM WITH 95% BOOTSTRAP CI")
print("=" * 75)
print(f"{'Condition':<12} {'Oracle IQM [95% CI]':<28} {'Learned IQM [95% CI]':<28}")
print("-" * 75)
summary = {}
rng = np.random.default_rng(12345)
for cond_name, _ in conditions:
    row = {}
    for mask_name, _ in mask_sources:
        central, lo, hi = stratified_bootstrap_ci(
            per_seed[(cond_name, mask_name)], n_boot=2000, rng=rng)
        row[mask_name] = (central, lo, hi)
        summary[(cond_name, mask_name)] = (central, lo, hi)
    print(f"{cond_name:<12} "
          f"{row['Oracle'][0]*100:5.1f} [{row['Oracle'][1]*100:4.1f},{row['Oracle'][2]*100:4.1f}]   "
          f"{row['Learned'][0]*100:5.1f} [{row['Learned'][1]*100:4.1f},{row['Learned'][2]*100:4.1f}]")

# ---- degradation ratios with CI ----
print()
print("=" * 75)
print("DEGRADATION RATIO (shifted / no-shift) WITH 95% CI")
print("=" * 75)
print(f"{'Condition':<12} {'Oracle':<22} {'Learned':<22}")
print("-" * 75)
for cond_name, _ in conditions[1:]:
    line = f"{cond_name:<12} "
    for mask_name, _ in mask_sources:
        base_lo, base_hi = summary[("No shift", mask_name)][1], summary[("No shift", mask_name)][2]
        sh_lo, sh_hi = summary[(cond_name, mask_name)][1], summary[(cond_name, mask_name)][2]
        base_mid = summary[("No shift", mask_name)][0]
        sh_mid = summary[(cond_name, mask_name)][0]
        ratio = sh_mid / base_mid if base_mid > 0 else float("nan")
        # crude CI on ratio (assumes independence)
        r_lo = sh_lo / base_hi if base_hi > 0 else float("nan")
        r_hi = sh_hi / base_lo if base_lo > 0 else float("nan")
        line += f"{ratio:.3f} [{r_lo:.2f},{r_hi:.2f}]   "
    print(line)

# ---- Welch's t-test: shifted vs no-shift, per mask ----
print()
print("=" * 75)
print("WELCH'S T-TEST (shifted vs no-shift, per mask)")
print("Bonferroni-corrected α = 0.05 / 8 = 0.00625")
print("=" * 75)
alpha_bonf = 0.05 / (len(conditions) - 1) * 2  # 8 tests total (4 conditions × 2 masks)
print(f"{'Condition':<12} {'Mask':<10} {'t':>8} {'p':>10} {'sig':>6}")
print("-" * 75)
for cond_name, _ in conditions[1:]:
    for mask_name, _ in mask_sources:
        base = np.concatenate(per_seed[("No shift", mask_name)])
        shifted = np.concatenate(per_seed[(cond_name, mask_name)])
        t, p = stats.ttest_ind(shifted, base, equal_var=False)
        sig = "***" if p < alpha_bonf else ("*" if p < 0.05 else "ns")
        print(f"{cond_name:<12} {mask_name:<10} {t:>8.3f} {p:>10.2e} {sig:>6}")
print("=" * 75)

STATISTICS RUN — 5 seeds × 100 episodes per condition × mask
  [No shift  ] [Oracle  ] mean= 73.4%  n=500
  [No shift  ] [Learned ] mean= 50.4%  n=500
  [Energy 0.5] [Oracle  ] mean= 62.0%  n=500
  [Energy 0.5] [Learned ] mean= 50.4%  n=500
  [Trust 0.8 ] [Oracle  ] mean= 48.8%  n=500
  [Trust 0.8 ] [Learned ] mean= 50.4%  n=500
  [Hop 5->2  ] [Oracle  ] mean= 67.4%  n=500
  [Hop 5->2  ] [Learned ] mean= 49.0%  n=500
  [Combined  ] [Oracle  ] mean= 46.6%  n=500
  [Combined  ] [Learned ] mean= 49.8%  n=500

IQM WITH 95% BOOTSTRAP CI
Condition    Oracle IQM [95% CI]          Learned IQM [95% CI]        
---------------------------------------------------------------------------
No shift      96.5 [89.6,100.0]    51.0 [43.2,58.4]
Energy 0.5    74.0 [67.6,80.0]    51.0 [43.2,58.4]
Trust 0.8     47.5 [41.6,54.0]    51.0 [43.2,58.4]
Hop 5->2      84.8 [77.6,92.4]    48.1 [39.2,56.0]
Combined      43.2 [34.0,51.6]    49.5 [41.2,57.6]

DEGRADATION RATIO (shifted / no-shift) WITH 95% CI
Conditi

In [80]:
print("=" * 70)
print("PER-SEED SUCCESS RATES (Oracle, No shift)")
print("=" * 70)
for s, vec in zip(SEEDS, per_seed[("No shift", "Oracle")]):
    print(f"  seed {s}: {vec.mean()*100:.1f}% (n={len(vec)})")

print()
print("=" * 70)
print("PER-SEED SUCCESS RATES (Learned, No shift)")
print("=" * 70)
for s, vec in zip(SEEDS, per_seed[("No shift", "Learned")]):
    print(f"  seed {s}: {vec.mean()*100:.1f}% (n={len(vec)})")

print()
print("=" * 70)
print("PER-SEED SUCCESS RATES (Oracle, Combined shift)")
print("=" * 70)
for s, vec in zip(SEEDS, per_seed[("Combined", "Oracle")]):
    print(f"  seed {s}: {vec.mean()*100:.1f}% (n={len(vec)})")

PER-SEED SUCCESS RATES (Oracle, No shift)
  seed 7000: 73.0% (n=100)
  seed 8000: 77.0% (n=100)
  seed 9000: 79.0% (n=100)
  seed 10000: 68.0% (n=100)
  seed 11000: 70.0% (n=100)

PER-SEED SUCCESS RATES (Learned, No shift)
  seed 7000: 54.0% (n=100)
  seed 8000: 51.0% (n=100)
  seed 9000: 56.0% (n=100)
  seed 10000: 44.0% (n=100)
  seed 11000: 47.0% (n=100)

PER-SEED SUCCESS RATES (Oracle, Combined shift)
  seed 7000: 50.0% (n=100)
  seed 8000: 51.0% (n=100)
  seed 9000: 51.0% (n=100)
  seed 10000: 42.0% (n=100)
  seed 11000: 39.0% (n=100)


In [81]:
def iqm(x):
    """Interquartile mean over per-run scores."""
    x = np.sort(x)
    n = len(x)
    lo = int(np.floor(0.25 * n))
    hi = int(np.ceil(0.75 * n))
    return float(np.mean(x[lo:hi])) if hi > lo else float(np.mean(x))


def iqm_from_runs(runs):
    """runs = list of per-seed success rates."""
    per_run = np.array([r.mean() for r in runs])
    return iqm(per_run)

In [82]:
# ============================================================
# CORRECTED STATISTICS — per-run IQM with bootstrap CI
# ============================================================
import numpy as np
from scipy import stats


def iqm(x):
    x = np.sort(x)
    n = len(x)
    lo = int(np.floor(0.25 * n))
    hi = int(np.ceil(0.75 * n))
    return float(np.mean(x[lo:hi])) if hi > lo else float(np.mean(x))


def bootstrap_iqm(runs, n_boot=2000, alpha=0.05, rng=None):
    rng = rng or np.random.default_rng(0)
    per_run = np.array([r.mean() for r in runs])
    n = len(per_run)
    samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        samples.append(iqm(per_run[idx]))
    samples = np.array(samples)
    return (float(np.mean(samples)),
            float(np.percentile(samples, 100 * alpha / 2)),
            float(np.percentile(samples, 100 * (1 - alpha / 2))))


print("=" * 75)
print("CORRECTED IQM (per-run) WITH 95% BOOTSTRAP CI")
print("=" * 75)
print(f"{'Condition':<12} {'Oracle':<26} {'Learned':<26}")
print("-" * 75)

summary = {}
rng = np.random.default_rng(12345)
for cond_name, _ in conditions:
    row = {}
    for mask_name, _ in mask_sources:
        central, lo, hi = bootstrap_iqm(per_seed[(cond_name, mask_name)],
                                        n_boot=2000, rng=rng)
        summary[(cond_name, mask_name)] = (central, lo, hi)
        row[mask_name] = (central, lo, hi)
    print(f"{cond_name:<12} "
          f"{row['Oracle'][0]*100:5.1f} [{row['Oracle'][1]*100:4.1f},{row['Oracle'][2]*100:4.1f}]   "
          f"{row['Learned'][0]*100:5.1f} [{row['Learned'][1]*100:4.1f},{row['Learned'][2]*100:4.1f}]")

print()
print("=" * 75)
print("CORRECTED DEGRADATION RATIO WITH 95% CI")
print("=" * 75)
print(f"{'Condition':<12} {'Oracle':<22} {'Learned':<22}")
print("-" * 75)
for cond_name, _ in conditions[1:]:
    line = f"{cond_name:<12} "
    for mask_name, _ in mask_sources:
        base_mid, base_lo, base_hi = summary[("No shift", mask_name)]
        sh_mid, sh_lo, sh_hi = summary[(cond_name, mask_name)]
        ratio = sh_mid / base_mid if base_mid > 0 else float("nan")
        r_lo = sh_lo / base_hi if base_hi > 0 else float("nan")
        r_hi = sh_hi / base_lo if base_lo > 0 else float("nan")
        line += f"{ratio:.3f} [{r_lo:.2f},{r_hi:.2f}]   "
    print(line)

print()
print("=" * 75)
print("PER-RUN SUCCESS RATES (for the paper appendix)")
print("=" * 75)
for cond_name, _ in conditions:
    for mask_name, _ in mask_sources:
        rates = [f"{r.mean()*100:.0f}" for r in per_seed[(cond_name, mask_name)]]
        print(f"  {cond_name:<12} {mask_name:<8} -> [{', '.join(rates)}]")
print("=" * 75)

CORRECTED IQM (per-run) WITH 95% BOOTSTRAP CI
Condition    Oracle                     Learned                   
---------------------------------------------------------------------------
No shift      73.5 [68.7,78.3]    50.7 [45.0,55.3]
Energy 0.5    62.1 [57.3,65.7]    50.6 [46.0,55.3]
Trust 0.8     48.5 [45.7,53.3]    50.6 [45.0,55.3]
Hop 5->2      67.4 [62.3,72.3]    49.3 [43.0,54.0]
Combined      47.1 [40.0,51.0]    49.9 [44.0,55.0]

CORRECTED DEGRADATION RATIO WITH 95% CI
Condition    Oracle                 Learned               
---------------------------------------------------------------------------
Energy 0.5   0.845 [0.73,0.96]   0.999 [0.83,1.23]   
Trust 0.8    0.660 [0.58,0.78]   0.999 [0.81,1.23]   
Hop 5->2     0.916 [0.80,1.05]   0.972 [0.78,1.20]   
Combined     0.641 [0.51,0.74]   0.984 [0.80,1.22]   

PER-RUN SUCCESS RATES (for the paper appendix)
  No shift     Oracle   -> [73, 77, 79, 68, 70]
  No shift     Learned  -> [54, 51, 56, 44, 47]
  Energy 0.5   Oracl

In [83]:
# ============================================================
# FIGURE GENERATION CELL
# Produces Figures 1, 3, 4, 5 as PDFs + data CSVs.
# ============================================================
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, csv

os.makedirs("/kaggle/working/paper_figures", exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 300,
})

# ---------------- Figure 1: Environment topology ----------------
env = WSNRoutingEnv(randomize_topology=False)
env.reset(seed=42)
fig, ax = plt.subplots(figsize=(4.2, 4.2))
pos = env.positions
# Draw edges
for i in range(env.n_nodes):
    for j in range(i + 1, env.n_nodes):
        if env.adj[i, j] > 0:
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                    color="lightgray", linewidth=0.4, zorder=1)
# Draw nodes
ax.scatter(pos[1:, 0], pos[1:, 1], s=18, c="steelblue", zorder=2, label="Sensor node")
ax.scatter(pos[0, 0], pos[0, 1], s=90, c="crimson", marker="s",
           zorder=3, label="Sink")
ax.set_xlim(0, env.area)
ax.set_ylim(0, env.area)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend(loc="upper right", frameon=True)
ax.set_title("WSN topology (50 nodes, transmission range = 25 m)")
plt.tight_layout()
plt.savefig("/kaggle/working/paper_figures/fig1_environment.pdf")
plt.close()

# ---------------- Figure 3: Success rate under shift ----------------
condition_names = [c[0] for c in conditions]
oracle_means, oracle_lo, oracle_hi = [], [], []
learned_means, learned_lo, learned_hi = [], [], []
for cond in condition_names:
    m, lo, hi = summary[(cond, "Oracle")]
    oracle_means.append(m * 100); oracle_lo.append(lo * 100); oracle_hi.append(hi * 100)
    m, lo, hi = summary[(cond, "Learned")]
    learned_means.append(m * 100); learned_lo.append(lo * 100); learned_hi.append(hi * 100)

x = np.arange(len(condition_names))
width = 0.36
fig, ax = plt.subplots(figsize=(6.5, 3.5))
err_o = [np.array(oracle_means) - np.array(oracle_lo),
         np.array(oracle_hi) - np.array(oracle_means)]
err_l = [np.array(learned_means) - np.array(learned_lo),
         np.array(learned_hi) - np.array(learned_means)]
ax.bar(x - width/2, oracle_means, width, yerr=err_o, capsize=3,
       color="#2E5C8A", label="Oracle mask", edgecolor="black", linewidth=0.5)
ax.bar(x + width/2, learned_means, width, yerr=err_l, capsize=3,
       color="#7FB069", label="Learned classifier mask", edgecolor="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(["No shift", "Energy\n0.5", "Trust\n0.8", "Hop\n5→2", "Combined"])
ax.set_ylabel("Success rate (%)")
ax.set_ylim(0, 90)
ax.axhline(0, color="black", linewidth=0.5)
ax.legend(loc="upper right", frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/kaggle/working/paper_figures/fig3_shift_bars.pdf")
plt.close()

# ---------------- Figure 4: Degradation ratios ----------------
shift_names = [c[0] for c in conditions[1:]]
oracle_ratios, oracle_r_lo, oracle_r_hi = [], [], []
learned_ratios, learned_r_lo, learned_r_hi = [], [], []
for cond in shift_names:
    bm, bl, bh = summary[("No shift", "Oracle")]
    sm, sl, sh = summary[(cond, "Oracle")]
    oracle_ratios.append(sm / bm)
    oracle_r_lo.append(sl / bh); oracle_r_hi.append(sh / bl)
    bm, bl, bh = summary[("No shift", "Learned")]
    sm, sl, sh = summary[(cond, "Learned")]
    learned_ratios.append(sm / bm)
    learned_r_lo.append(sl / bh); learned_r_hi.append(sh / bl)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
x = np.arange(len(shift_names))
ax.plot(x, oracle_ratios, marker="o", color="#2E5C8A", label="Oracle mask", linewidth=1.8)
ax.fill_between(x, oracle_r_lo, oracle_r_hi, color="#2E5C8A", alpha=0.18)
ax.plot(x, learned_ratios, marker="s", color="#7FB069", label="Learned classifier mask", linewidth=1.8)
ax.fill_between(x, learned_r_lo, learned_r_hi, color="#7FB069", alpha=0.18)
ax.axhline(1.0, color="black", linewidth=0.7, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(["Energy 0.5", "Trust 0.8", "Hop 5→2", "Combined"])
ax.set_ylabel("Degradation ratio\n(shifted / no-shift)")
ax.set_ylim(0.3, 1.3)
ax.legend(loc="lower left", frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/kaggle/working/paper_figures/fig4_degradation.pdf")
plt.close()

# ---------------- Figure 5: Training curves ----------------
# Extract from TensorBoard logs if available
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    import glob
    log_dirs = glob.glob("/kaggle/working/ppo_hard_mask/**/events.out.tfevents.*", recursive=True)
    if log_dirs:
        ea = EventAccumulator(log_dirs[0], size_guidance={"scalars": 0})
        ea.Reload()
        tags = ea.Tags().get("scalars", [])
        rew_tag = [t for t in tags if "ep_rew_mean" in t]
        len_tag = [t for t in tags if "ep_len_mean" in t]
        fig, ax1 = plt.subplots(figsize=(6.5, 3.5))
        if rew_tag:
            events = ea.Scalars(rew_tag[0])
            steps = [e.step for e in events]
            vals = [e.value for e in events]
            ax1.plot(steps, vals, color="#2E5C8A", linewidth=1.5, label="ep_rew_mean")
            ax1.set_xlabel("Timesteps")
            ax1.set_ylabel("Mean episode reward", color="#2E5C8A")
            ax1.tick_params(axis="y", labelcolor="#2E5C8A")
        ax2 = ax1.twinx()
        if len_tag:
            events = ea.Scalars(len_tag[0])
            steps = [e.step for e in events]
            vals = [e.value for e in events]
            ax2.plot(steps, vals, color="#7FB069", linewidth=1.5, linestyle="--", label="ep_len_mean")
            ax2.set_ylabel("Mean episode length", color="#7FB069")
            ax2.tick_params(axis="y", labelcolor="#7FB069")
        ax1.grid(axis="x", linestyle="--", alpha=0.3)
        plt.title("PPO training progress")
        plt.tight_layout()
        plt.savefig("/kaggle/working/paper_figures/fig5_training.pdf")
        plt.close()
        print("Fig 5: generated from TensorBoard log")
    else:
        print("Fig 5: no TensorBoard log found — skip for now")
except Exception as e:
    print(f"Fig 5: skipped ({type(e).__name__}: {e})")

# ---------------- Save data CSVs for reproducibility ----------------
with open("/kaggle/working/paper_figures/results_data.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["condition", "mask", "iqm", "ci_lo", "ci_hi"])
    for (cond, mask), (m, lo, hi) in summary.items():
        w.writerow([cond, mask, f"{m:.4f}", f"{lo:.4f}", f"{hi:.4f}"])

with open("/kaggle/working/paper_figures/per_seed_data.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["condition", "mask", "seed_idx", "success_rate"])
    for (cond, mask), runs in per_seed.items():
        for i, r in enumerate(runs):
            w.writerow([cond, mask, i, f"{r.mean():.4f}"])

print()
print("=" * 65)
print("Figures written to /kaggle/working/paper_figures/")
print("=" * 65)
print(os.listdir("/kaggle/working/paper_figures"))

Fig 5: generated from TensorBoard log

Figures written to /kaggle/working/paper_figures/
['results_data.csv', 'fig3_shift_bars.pdf', 'fig4_degradation.pdf', 'fig1_environment.pdf', 'fig5_training.pdf', 'per_seed_data.csv']


In [84]:
import os, shutil, json, subprocess

PAPER = "/kaggle/working/paper_supplementary"
os.makedirs(PAPER, exist_ok=True)

# ---- Models ----
for src in ["/kaggle/working/ppo_hard_mask/ppo_hard_mask.zip",
            "/kaggle/working/feasibility_classifier.pt"]:
    if os.path.exists(src):
        shutil.copy(src, PAPER)

# ---- Figures ----
fig_dir = "/kaggle/working/paper_figures"
if os.path.isdir(fig_dir):
    shutil.copytree(fig_dir, os.path.join(PAPER, "figures"), dirs_exist_ok=True)

# ---- Notebook ----
# The notebook file is saved as __notebook__.ipynb in the working dir when committed.
# Also try to find the current .ipynb
for candidate in ["/kaggle/working/__notebook__.ipynb"]:
    if os.path.exists(candidate):
        shutil.copy(candidate, os.path.join(PAPER, "notebook.ipynb"))
        break

# ---- requirements.txt ----
with open(os.path.join(PAPER, "requirements.txt"), "w") as f:
    f.write(
        "stable-baselines3==2.9.0\n"
        "gymnasium==1.2.0\n"
        "torch==2.11.0\n"
        "numpy==2.1.3\n"
        "scipy==1.16.3\n"
        "matplotlib==3.8.2\n"
    )

# ---- README.md ----
readme = """# Frozen Feasibility Priors — Supplementary Code

## Overview
Code and data for the paper:
"Frozen Feasibility Priors: Learned Masks Provide Robustness to Constraint Shift
in Constrained Reinforcement Learning for Wireless Sensor Networks"

## Files
- `notebook.ipynb` — full pipeline: environment, PPO training, classifier training, shift evaluation, statistics
- `ppo_hard_mask.zip` — trained PPO policy (SB3 format)
- `feasibility_classifier.pt` — trained feasibility classifier (PyTorch state dict)
- `figures/` — publication figures (PDF)
- `results_data.csv` — IQM + bootstrap CI table
- `per_seed_data.csv` — per-seed success rates
- `requirements.txt` — Python dependencies

## Reproduction
1. Install dependencies: `pip install -r requirements.txt`
2. Open `notebook.ipynb` in Jupyter or Kaggle
3. Run cells top-to-bottom. The random seeds used are `[7000, 8000, 9000, 10000, 11000]`.
4. Full training + evaluation completes in ~1 hour on a CPU-only machine.

## Environment
- Python 3.13
- Kaggle notebook (CPU runtime)
- No proprietary data or hardware required

## License
Code: MIT
Data: CC-BY 4.0
"""
with open(os.path.join(PAPER, "README.md"), "w") as f:
    f.write(readme)

# ---- Zip everything ----
shutil.make_archive("/kaggle/working/paper_supplementary",
                    "zip", "/kaggle/working/paper_supplementary")

# ---- List ----
print("=" * 65)
print("SUPPLEMENTARY CONTENTS")
print("=" * 65)
for root, dirs, files in os.walk(PAPER):
    for f in files:
        full = os.path.join(root, f)
        rel = os.path.relpath(full, PAPER)
        size = os.path.getsize(full) / 1024
        print(f"  {rel:<50} {size:>8.1f} KB")

print()
print("ZIP created:", os.path.getsize("/kaggle/working/paper_supplementary.zip") / 1e6, "MB")

SUPPLEMENTARY CONTENTS
  feasibility_classifier.pt                              22.3 KB
  ppo_hard_mask.zip                                     632.1 KB
  README.md                                               1.1 KB
  requirements.txt                                        0.1 KB
  figures/results_data.csv                                0.4 KB
  figures/fig3_shift_bars.pdf                            18.5 KB
  figures/fig4_degradation.pdf                           17.1 KB
  figures/fig1_environment.pdf                           16.1 KB
  figures/fig5_training.pdf                              13.6 KB
  figures/per_seed_data.csv                               1.4 KB

ZIP created: 0.520415 MB
